In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: optional
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# LoRA từ số 0 bằng NumPy

`W` bị đóng băng; chỉ học `A` và `B`, với `ΔW=(alpha/r)BA`. Khởi tạo B=0 giữ output ban đầu.

In [ ]:
class LoRALinear:
    """Frozen W plus scaled low-rank update B@A."""
    def __init__(self,W,rank=2,alpha=4,seed=42):
        self.W=np.asarray(W,float).copy(); self.rank=rank; self.scale=alpha/rank
        rng=np.random.default_rng(seed); self.A=rng.normal(0,.01,(rank,self.W.shape[1])); self.B=np.zeros((self.W.shape[0],rank))
    def __call__(self,x): return x@self.W.T + self.scale*(x@self.A.T)@self.B.T
    def merged_weight(self): return self.W+self.scale*self.B@self.A

rng=np.random.default_rng(1); W=rng.normal(size=(6,8)); x=rng.normal(size=(4,8)); layer=LoRALinear(W,rank=2)
assert np.allclose(layer(x),x@W.T)
layer.B[:]=rng.normal(0,.01,layer.B.shape)
assert np.allclose(layer(x),x@layer.merged_weight().T)
base=W.size; trainable=layer.A.size+layer.B.size
assert trainable==2*(8+6); print({"base":base,"lora":trainable,"ratio":trainable/base})